In [81]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq

In [ ]:

llm = ChatGroq(model="openai/gpt-oss-120b",api_key="OOOO")

In [83]:
class TextState(TypedDict):
    raw_text:str
    analysis:str
    final_summary:str

In [84]:
def analyze_node(state:TextState):
    text=state['raw_text']
    prompt=f"Analyze the tone and key points of this text in 3 bullet point \n{text}"
    response=llm.invoke(prompt)
    return{"analysis":response.content}

In [85]:
def summarize_node(state:TextState):
    current_anaysis=state['analysis']
    prompt=f'Based on this analysis , write a 1 sentace funny +  crisp summary \n {current_anaysis}'
    response=llm.invoke(prompt)

    return {"final_summary":response.content}

In [86]:
builder=StateGraph(TextState)


In [87]:
builder.add_node("analyzer",analyze_node)
builder.add_node("summarizer",summarize_node)


In [88]:
builder.add_edge(START,'analyzer')
builder.add_edge('analyzer','summarizer')
builder.add_edge('summarizer',END)

In [89]:
graph=builder.compile()

In [90]:
input_data={
    'raw_text':'I am wasting my time playing games and because of gaming I am unable to manage my time properly.'
}
result=graph.invoke(input_data)

In [94]:
print(result["analysis"])
print(result["final_summary"])

- **Tone:** Self‑critical and reflective, expressing regret and frustration about personal habits.  
- **Key point 1:** The speaker feels that gaming is a waste of time.  
- **Key point 2:** Excessive gaming is interfering with effective time management and daily responsibilities.
I’ve become the world‑class champion of turning every to‑do list into a high‑score leaderboard, proving that my gaming habit is the most efficient way to waste time and sabotage my own schedule.
